# 01 — Data Understanding

This review defines the prediction contract, checks current data and building-split evidence, and records feature-availability decisions. Values are refreshed from project artifacts at execution time.

## Problem contract

### Prediction unit and timing

This is binary point-anomaly classification for an observed hourly meter record. One observation is keyed by `(building_id, timestamp)`. The prediction concerns whether that already-observed hour is anomalous; it is not a forecast of a future reading or future anomaly. The target is `anomaly`, with `1` denoting the positive class.

### Evaluation and decision boundary

ROC-AUC is the primary competition metric. Evaluation preserves continuous anomaly-ranking scores rather than converting them to hard labels. False-positive and false-negative operating costs are unspecified, so no operational threshold is optimized. Any eventual competition file is uploaded manually by the participant; this notebook does not create or upload one. Training and test buildings are expected to be disjoint, matching the unseen-building evaluation setting.

## Reproducibility and raw data

### Portable inputs

Raw competition files are read from `data/energy-anomaly-detection/`. Repository-relative paths resolve from the project root or this notebook's directory. Original data remain read-only; derived evidence belongs under `outputs/`. The compact inventory uses current schemas and persisted preparation lineage without printing source rows or machine-specific paths.

**Purpose:** Resolve project paths, inspect lazy raw-table schemas, and check for current preparation, split, and shift artifacts.

In [1]:
import json
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

from project.data.load import scan_all_raw_tables
from project.data.validate import compute_file_sha256
from project.paths import ProjectPaths

paths = ProjectPaths.resolve(start=Path.cwd())
raw_tables = scan_all_raw_tables(paths)
lineage_path = paths.lineage_dir / 'row_lineage.json'
train_joined_path = paths.prepared_dir / 'train_joined.parquet'
test_joined_path = paths.prepared_dir / 'test_joined.parquet'
split_path = paths.splits_dir / 'building_folds.parquet'
split_metadata_path = paths.splits_dir / 'split_metadata.json'
shift_json_path = paths.shift_dir / 'adversarial_validation.json'
domain_oof_path = paths.shift_dir / 'domain_oof.parquet'
importance_path = paths.shift_dir / 'fold_averaged_importance.csv'
artifacts = {'join lineage': lineage_path, 'train join': train_joined_path,
            'test join': test_joined_path, 'building manifest': split_path,
            'split metadata': split_metadata_path, 'domain summary': shift_json_path,
            'domain OOF': domain_oof_path, 'fold importance': importance_path}
missing_artifacts = [name for name, path in artifacts.items() if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError(f'Required project artifacts are missing: {missing_artifacts}')
lineage = json.loads(lineage_path.read_text(encoding='utf-8'))
raw_schemas = {name: frame.collect_schema() for name, frame in raw_tables.items()}
train_joined = pl.scan_parquet(train_joined_path)
test_joined = pl.scan_parquet(test_joined_path)
train_schema = train_joined.collect_schema()
test_schema = test_joined.collect_schema()
raw_metadata = lineage.get('raw_file_metadata', {})
source_inventory = pl.DataFrame([
    {'source': name, 'rows_from_lineage': raw_metadata.get(f'{name}.csv', {}).get('rows'),
     'column_count': len(schema),
     'has_building_time_key': {'building_id', 'timestamp'}.issubset(schema.names()),
     'has_row_id': 'row_id' in schema, 'has_target': 'anomaly' in schema,
     'gte_family_count': sum(column.startswith('gte_') for column in schema.names())}
    for name, schema in raw_schemas.items()
]).sort('source')
display(source_inventory)

source,rows_from_lineage,column_count,has_building_time_key,has_row_id,has_target,gte_family_count
str,i64,i64,bool,bool,bool,i64
"""sample_submission""",1800567,2,false,true,true,0
"""test""",1800567,4,true,true,false,0
"""test_features""",1800567,57,true,true,false,16
"""train""",1749494,4,true,false,true,0
"""train_features""",1749494,57,true,false,true,16


**Observation:** The inventory reflects current raw schemas and preparation lineage.

**Interpretation:** Schema presence establishes shape, not prediction-time validity or feature provenance.

**Action:** Keep source files unchanged and use the join and availability checks below before modeling decisions.

## Join and data integrity

Training joins use `(building_id, timestamp)` and test joins use `row_id`; test order must match the sample file. Preparation lineage records unique keys, one-to-one joins, equality of duplicate `meter_reading` and training-target copies, building separation, and sample order. Current persisted Parquet artifacts are checked with compact streaming diagnostics.

**Purpose:** Verify persisted join assertions, key uniqueness, building disjointness, row totals, and sample order without displaying joined rows.

In [2]:
def duplicate_key_count(frame: pl.LazyFrame, keys: list[str]) -> int:
    return int(frame.group_by(keys).len().filter(pl.col('len') > 1)
               .select(pl.len()).collect(engine='streaming').item())

checks = lineage.get('validations', {})
required_checks = ('unique_keys_validated', 'one_to_one_joins_validated',
                  'duplicate_copy_equality_validated', 'sample_submission_order_preserved')
failed_checks = [name for name in required_checks if checks.get(name) is not True]
if checks.get('train_test_building_overlap') != 0:
    failed_checks.append('train_test_building_overlap')
if failed_checks:
    raise ValueError(f'Persisted join checks failed: {failed_checks}')
train_rows = int(train_joined.select(pl.len()).collect(engine='streaming').item())
test_rows = int(test_joined.select(pl.len()).collect(engine='streaming').item())
train_duplicate_keys = duplicate_key_count(train_joined, ['building_id', 'timestamp'])
test_duplicate_keys = duplicate_key_count(test_joined, ['row_id'])
building_overlap = int(train_joined.select('building_id').unique()
    .join(test_joined.select('building_id').unique(), on='building_id', how='inner')
    .select(pl.len()).collect(engine='streaming').item())
test_order = test_joined.select('row_id').with_row_index('_position')
sample_order = raw_tables['sample_submission'].select('row_id').with_row_index('_position')
order_mismatches = int(test_order.join(sample_order, on='_position', suffix='_sample')
    .filter(pl.col('row_id') != pl.col('row_id_sample'))
    .select(pl.len()).collect(engine='streaming').item())
if train_duplicate_keys or test_duplicate_keys or building_overlap or order_mismatches:
    raise ValueError('Current prepared joins violate a key, building, or ordering invariant')
if 'anomaly' not in train_schema or 'anomaly' in test_schema:
    raise ValueError('Target must be present in training and absent from test')
join_audit = pl.DataFrame([
    {'check': 'train rows', 'observed': train_rows, 'status': 'verified'},
    {'check': 'test rows', 'observed': test_rows, 'status': 'verified'},
    {'check': 'duplicate train keys', 'observed': train_duplicate_keys, 'status': 'verified'},
    {'check': 'duplicate test keys', 'observed': test_duplicate_keys, 'status': 'verified'},
    {'check': 'overlapping buildings', 'observed': building_overlap, 'status': 'verified'},
    {'check': 'test/sample order mismatches', 'observed': order_mismatches, 'status': 'verified'},
    {'check': 'train feature columns', 'observed': len(train_schema), 'status': 'verified'},
    {'check': 'test feature columns', 'observed': len(test_schema), 'status': 'verified'}
])
display(join_audit)

check,observed,status
str,i64,str
"""train rows""",1749494,"""verified"""
"""test rows""",1800567,"""verified"""
"""duplicate train keys""",0,"""verified"""
"""duplicate test keys""",0,"""verified"""
"""overlapping buildings""",0,"""verified"""
"""test/sample order mismatches""",0,"""verified"""
"""train feature columns""",57,"""verified"""
"""test feature columns""",57,"""verified"""


**Observation:** The table reports current join and key diagnostics; duplicate-copy equality comes from validated preparation lineage.

**Interpretation:** A passing join contract supports row alignment but does not prove every joined feature is safe at scoring time.

**Action:** Treat any failed invariant as a data-preparation blocker before interpreting model comparisons.

## Target and data quality

### Target support and feature quality

Target support is summarized from the current training join. Feature missingness is reported overall and by building and calendar month. True constants and highly concentrated quasi-constants are diagnostic flags only: a constant may be removed later only by a training-fold-fitted mask, while a quasi-constant requires a cross-validation ablation. The concentration cutoff is a review heuristic, not a learned preprocessing rule.

Anomalous labels and unusual meter readings are retained as possible signal. No observation is silently deleted, winsorized, or capped here.

**Purpose:** Summarize current target prevalence, feature missingness, distinct values, concentration flags, and building/month missingness ranges.

In [3]:
target_summary = train_joined.group_by('anomaly').agg(pl.len().alias('rows'))
target_summary = target_summary.sort('anomaly').collect(engine='streaming')
if set(target_summary.get_column('anomaly').to_list()) != {0, 1}:
    raise ValueError('Training target support must contain both binary classes')
target_summary = target_summary.with_columns((pl.col('rows') / pl.col('rows').sum()).alias('row_share'))
display(target_summary)
excluded = {'building_id', 'timestamp', 'row_id', 'anomaly'}
feature_columns = [name for name in train_schema.names() if name not in excluded]
if not feature_columns:
    raise ValueError('No feature columns remain after excluding keys and target')
quasi_constant_cutoff = 0.995
quality_expr = []
for feature in feature_columns:
    value = pl.col(feature)
    if train_schema[feature].is_float():
        value = value.fill_nan(None)
    mode = value.mode().first()
    quality_expr.extend([value.is_null().mean().alias(f'{feature}__missing'),
                         value.drop_nulls().n_unique().alias(f'{feature}__unique'),
                         ((value == mode).sum() / value.count()).alias(f'{feature}__dominant')])
quality_values = train_joined.select(quality_expr).collect(engine='streaming').row(0, named=True)
def missing_by_group(grouped: pl.LazyFrame, group: str) -> dict[str, object]:
    rates = [(pl.col(c).fill_nan(None) if train_schema[c].is_float() else pl.col(c))
             .is_null().mean().alias(c) for c in feature_columns]
    values = grouped.group_by(group).agg(rates).select([pl.col(c).max().alias(c) for c in feature_columns])
    return values.collect(engine='streaming').row(0, named=True)
building_missing = missing_by_group(train_joined, 'building_id')
monthly_missing = missing_by_group(train_joined.with_columns(
    pl.col('timestamp').dt.truncate('1mo').alias('_month')), '_month')
quality_rows = []
for feature in feature_columns:
    unique_count = int(quality_values[f'{feature}__unique'])
    dominant = quality_values[f'{feature}__dominant']
    status = ('all-missing' if unique_count == 0 else 'true constant' if unique_count == 1
              else 'quasi-constant' if dominant is not None and dominant >= quasi_constant_cutoff
              else 'variable')
    quality_rows.append({'feature': feature, 'missing_rate': quality_values[f'{feature}__missing'],
        'max_building_missing_rate': building_missing[feature],
        'max_month_missing_rate': monthly_missing[feature],
        'non_null_unique_count': unique_count, 'dominant_non_null_share': dominant, 'status': status})
feature_quality = pl.DataFrame(quality_rows)
display(feature_quality.sort('missing_rate', descending=True).head(15))
display(feature_quality.filter(pl.col('status').is_in(
    ['true constant', 'quasi-constant', 'all-missing'])).sort(['status', 'feature']))

anomaly,rows,row_share
i64,u32,f64
0,1712198,0.978682
1,37296,0.021318


feature,missing_rate,max_building_missing_rate,max_month_missing_rate,non_null_unique_count,dominant_non_null_share,status
str,f64,f64,f64,i64,f64,str
"""meter_reading""",0.061534,0.615893,0.106471,191681,0.0,"""variable"""
"""site_id""",0.0,0.0,0.0,14,0.260234,"""variable"""
"""primary_use""",0.0,0.0,0.0,12,0.400567,"""variable"""
"""square_feet""",0.0,0.0,0.0,200,0.005021,"""variable"""
"""year_built""",0.0,0.0,0.0,32,0.663678,"""variable"""
…,…,…,…,…,…,…
"""sea_level_pressure""",0.0,0.0,0.0,710,0.135644,"""variable"""
"""wind_direction""",0.0,0.0,0.0,44,0.125105,"""variable"""
"""wind_speed""",0.0,0.0,0.0,59,0.125222,"""variable"""


feature,missing_rate,max_building_missing_rate,max_month_missing_rate,non_null_unique_count,dominant_non_null_share,status
str,f64,f64,f64,i64,f64,str
"""gte_meter""",0.0,0.0,0.0,1,1.0,"""true constant"""
"""year""",0.0,0.0,0.0,1,1.0,"""true constant"""


**Observation:** The tables show current class support, missingness ranges, and feature-concentration flags.

**Interpretation:** Dataset-wide constant diagnostics do not authorize dropping a column from a validation fold; missingness and unusual meter values may carry signal.

**Action:** Preserve anomalous rows. Defer imputation, feature removal, and outlier treatment to explicitly compared, fold-local experiments.

## Train/test shift

### Per-building domain diagnostic

The persisted diagnostic uses one summary row per building so repeated hourly observations are not treated as independent buildings. Numeric fields are summarized by mean, median, and missingness; categorical fields by mode only. Building row count is diagnostic metadata only. Building identifiers, source fields, targets, and domain labels are not classifier predictors. This notebook reads script-produced ROC-AUC, Average Precision, LogLoss, fold summaries, null references, OOF scores, and fold-averaged importance; it does not fit a domain classifier.

For ROC-AUC values at or above 0.5, the bands `<=0.55`, `0.55–0.60`, and `>0.60` are heuristic descriptions of weak, moderate, and more material train/test distinguishability, not automatic decisions. AUC below 0.5 for the declared positive class `test=1` is reported separately as an inverse ranking that requires class/score-orientation and fold-level review; do not invert scores after observing the result. Material shift triggers validation, robustness, and monitoring review. It never authorizes test-guided feature deletion, feature selection, or production-preprocessing fitting.

**Purpose:** Read current domain-validation metrics and artifact schemas, plus compact univariate train/test shift summaries.

In [4]:
domain_report = json.loads(shift_json_path.read_text(encoding='utf-8'))
def validate_domain_artifact_hashes(
    report: dict[str, object], artifact_paths: dict[str, Path]
) -> None:
    if not isinstance(report, dict):
        raise TypeError('Domain validation report must be a JSON object')
    recorded_artifacts = report.get('artifacts')
    if not isinstance(recorded_artifacts, dict):
        raise TypeError('Domain report is missing its artifact hash manifest')
    for artifact_name, artifact_path in artifact_paths.items():
        artifact = recorded_artifacts.get(artifact_name)
        expected_hash = artifact.get('sha256') if isinstance(artifact, dict) else None
        if not isinstance(expected_hash, str) or not expected_hash:
            raise ValueError(
                f'Domain report is missing the SHA-256 for {artifact_name}'
            )
        if compute_file_sha256(artifact_path) != expected_hash:
            raise ValueError(
                f'Domain report has a stale hash for {artifact_name}; '
                'rerun adversarial validation'
            )

validate_domain_artifact_hashes(domain_report, {
    'input_train_parquet': train_joined_path,
    'input_test_parquet': test_joined_path,
    'domain_oof_parquet': domain_oof_path,
    'fold_averaged_importance_csv': importance_path,
})
def scalar_metrics(payload: object, prefix: str = '') -> list[dict[str, object]]:
    terms = ('auc', 'average_precision', 'log_loss', 'prevalence', 'null', 'entropy', 'fold')
    rows: list[dict[str, object]] = []
    if isinstance(payload, dict):
        for key, value in payload.items():
            name = f'{prefix}.{key}' if prefix else str(key)
            rows.extend(scalar_metrics(value, name))
    elif isinstance(payload, list):
        for index, value in enumerate(payload):
            rows.extend(scalar_metrics(value, f'{prefix}[{index}]'))
    elif (isinstance(payload, (int, float)) and not isinstance(payload, bool)
          and any(term in prefix.casefold() for term in terms)):
        rows.append({'metric': prefix, 'value': float(payload)})
    return rows
domain_metrics = pl.DataFrame(scalar_metrics(domain_report),
    schema={'metric': pl.String, 'value': pl.Float64})
display(domain_metrics.sort('metric'))
domain_oof = pl.scan_parquet(domain_oof_path)
domain_oof_schema = domain_oof.collect_schema()
domain_oof_rows = int(domain_oof.select(pl.len()).collect(engine='streaming').item())
display(pl.DataFrame([{'artifact': 'domain_oof.parquet', 'rows': domain_oof_rows,
    'columns': len(domain_oof_schema), 'has_fold_field': 'fold' in domain_oof_schema}]))
fold_importance = pl.read_csv(importance_path)
importance_column = next((c for c in ('mean_importance', 'importance_mean', 'importance',
    'fold_averaged_importance') if c in fold_importance.columns), None)
feature_name_column = next((c for c in ('feature', 'feature_name', 'column')
    if c in fold_importance.columns), None)
if importance_column and feature_name_column:
    display(fold_importance.sort(importance_column, descending=True)
        .select(feature_name_column, importance_column).head(12))
else:
    display(pl.DataFrame({'importance_artifact_columns': fold_importance.columns}))
numeric_features = [name for name in feature_columns
    if name in test_schema and train_schema[name].is_numeric()]
def numeric_summary(frame: pl.LazyFrame) -> dict[str, object]:
    expressions = []
    for feature in numeric_features:
        value = pl.col(feature)
        if train_schema[feature].is_float():
            value = value.fill_nan(None)
        expressions.extend([value.mean().alias(f'{feature}__mean'),
            value.median().alias(f'{feature}__median'), value.std().alias(f'{feature}__std'),
            value.is_null().mean().alias(f'{feature}__missing')])
    return frame.select(expressions).collect(engine='streaming').row(0, named=True)
train_numeric = numeric_summary(train_joined)
test_numeric = numeric_summary(test_joined)
shift_rows = []
for feature in numeric_features:
    train_mean, test_mean = train_numeric[f'{feature}__mean'], test_numeric[f'{feature}__mean']
    train_std = train_numeric[f'{feature}__std']
    shift_rows.append({'feature': feature, 'train_mean': train_mean, 'test_mean': test_mean,
        'mean_difference': test_mean - train_mean,
        'mean_difference_in_train_sd': (test_mean - train_mean) / train_std
            if train_std is not None and train_std > 0 else None,
        'train_median': train_numeric[f'{feature}__median'],
        'test_median': test_numeric[f'{feature}__median'],
        'train_missingness': train_numeric[f'{feature}__missing'],
        'test_missingness': test_numeric[f'{feature}__missing']})
univariate_shift = pl.DataFrame(shift_rows)
if univariate_shift.height:
    display(univariate_shift.sort('mean_difference_in_train_sd',
        descending=True, nulls_last=True).head(12))

metric,value
str,f64
"""configuration.folds""",5.0
"""metrics.fold_mean_std.average_…",0.472989
"""metrics.fold_mean_std.average_…",0.034783
"""metrics.fold_mean_std.log_loss…",0.716909
"""metrics.fold_mean_std.log_loss…",0.013947
…,…
"""metrics.pooled_oof.log_loss""",0.716883
"""metrics.pooled_oof.roc_auc""",0.435267
"""null_baselines.average_precisi…",0.507389


artifact,rows,columns,has_fold_field
str,i64,i64,bool
"""domain_oof.parquet""",406,5,true


feature,importance_mean
str,f64
"""meter_reading__mean""",0.012086
"""gte_meter_building_id_weekday_…",0.008531
"""meter_reading__median""",0.007424
"""gte_meter_building_id_month__m…",0.006981
"""air_temperature_std_lag73__mea…",0.006314
…,…
"""square_feet__median""",0.004412
"""gte_meter_building_id_hour__me…",0.004322
"""air_temperature_std_lag73__med…",0.003832


feature,train_mean,test_mean,mean_difference,mean_difference_in_train_sd,train_median,test_median,train_missingness,test_missingness
str,f64,f64,f64,f64,f64,f64,f64,f64
"""year_built""",192.55816,195.356921,2.798762,0.031326,255.0,255.0,0.0,0.0
"""gte_site_id""",4.380949,4.402207,0.021258,0.027463,4.71,4.71,0.0,0.0
"""site_id""",7.819728,7.912157,0.092429,0.019136,9.0,9.0,0.0,0.0
"""air_temperature_std_lag73""",3.647699,3.663056,0.015356,0.012248,3.621,3.641,0.0,0.0
"""gte_meter_site_id""",4.261428,4.268292,0.006864,0.009921,4.548,4.548,0.0,0.0
…,…,…,…,…,…,…,…,…
"""air_temperature_std_lag7""",1.588481,1.593044,0.004563,0.004272,1.361,1.362,0.0,0.0
"""air_temperature_max_lag73""",22.479426,22.515292,0.035865,0.003609,23.234,23.234,0.0,0.0
"""air_temperature_max_lag7""",18.423979,18.437669,0.01369,0.001306,18.719,18.719,0.0,0.0


**Observation:** Persisted OOF metrics and univariate summaries describe train/test distinguishability.

**Interpretation:** The univariate comparison is row-weighted; the domain diagnostic uses building-level summaries. Differences and importance are shift evidence only, not feature-selection criteria. Test predictors must not fit production preprocessing.

**Action:** Treat AUC below 0.5 as an inverse-ranking signal requiring orientation and fold-level review; do not invert it after observing results. For a positive-direction score in the material heuristic band, review validation design, robustness coverage, and monitoring. Do not use test predictors to select features.

## Feature availability and leakage

A feature is eligible for a later predictor contract only after its generating source, target dependence, aggregation scope, and availability at the observed-hour prediction cutoff are verified. `building_id` is a grouping key only and is not a raw predictor. `row_id`, target, fold/domain indicators, and source metadata are not predictors. Future-derived, target-derived, and unverified features remain excluded until their lineage is demonstrated.

Every current `gte_*` field is `pending/blocked` because no verified provenance record is available in the current feature metadata. The persisted domain-shift report may include these values strictly for shift diagnosis; they remain blocked as production predictors and are not selected from test results. This status changes only when source, target dependence, aggregation boundary, and time availability are verified. Lag and building-aggregate fields also require prediction-time and fold-boundary review; their names alone do not establish safety.

**Purpose:** Inventory discovered `gte_*` fields and show the identifier, target, and metadata exclusion policy.

In [5]:
gte_columns = [name for name in train_schema.names() if name.startswith('gte_')]
gte_audit = pl.DataFrame([
    {'feature': feature, 'provenance_status': 'pending/blocked',
     'production_predictor_eligible': False,
     'required_evidence': 'source, target dependence, aggregation scope, prediction-time availability'}
    for feature in gte_columns
], schema={'feature': pl.String, 'provenance_status': pl.String,
          'production_predictor_eligible': pl.Boolean, 'required_evidence': pl.String})
display(gte_audit)
feature_policy = pl.DataFrame([
    {'field': 'building_id', 'role': 'grouping key only', 'production_predictor_policy': 'exclude raw identifier'},
    {'field': 'row_id', 'role': 'row alignment key', 'production_predictor_policy': 'exclude'},
    {'field': 'anomaly', 'role': 'training target', 'production_predictor_policy': 'exclude from X'},
    {'field': 'fold/domain/source metadata', 'role': 'validation or lineage metadata', 'production_predictor_policy': 'exclude from X'},
    {'field': 'gte_*', 'role': 'unverified feature family', 'production_predictor_policy': 'pending/blocked'}
])
display(feature_policy)

feature,provenance_status,production_predictor_eligible,required_evidence
str,str,bool,str
"""gte_hour""","""pending/blocked""",false,"""source, target dependence, agg…"
"""gte_weekday""","""pending/blocked""",false,"""source, target dependence, agg…"
"""gte_month""","""pending/blocked""",false,"""source, target dependence, agg…"
"""gte_building_id""","""pending/blocked""",false,"""source, target dependence, agg…"
"""gte_primary_use""","""pending/blocked""",false,"""source, target dependence, agg…"
…,…,…,…
"""gte_meter_primary_use""","""pending/blocked""",false,"""source, target dependence, agg…"
"""gte_meter_site_id""","""pending/blocked""",false,"""source, target dependence, agg…"
"""gte_meter_building_id_hour""","""pending/blocked""",false,"""source, target dependence, agg…"


field,role,production_predictor_policy
str,str,str
"""building_id""","""grouping key only""","""exclude raw identifier"""
"""row_id""","""row alignment key""","""exclude"""
"""anomaly""","""training target""","""exclude from X"""
"""fold/domain/source metadata""","""validation or lineage metadata""","""exclude from X"""
"""gte_*""","""unverified feature family""","""pending/blocked"""


**Observation:** The current schema contains the fields listed above; the `gte_*` status is explicit and conservative.

**Interpretation:** Availability cannot be inferred from a feature name or train/test distribution comparison.

**Action:** Exclude unverified families and identifiers from model inputs until fold-safe provenance and availability evidence is recorded.

## Persisted split decision

The training assignment is loaded from `outputs/splits/building_folds.parquet`; this notebook does not create or reshuffle a split. One manifest row represents one training building. The manifest must cover current training buildings exactly, remain disjoint from test buildings, and retain positive and negative support within each persisted partition. Its artifact hash is checked against the accompanying split metadata.

**Purpose:** Verify the persisted manifest hash, building membership, group separation, and class support by partition.

In [6]:
split_manifest = pl.read_parquet(split_path)
split_metadata = json.loads(split_metadata_path.read_text(encoding='utf-8'))
required_manifest_columns = {'building_id', 'split', 'fold', 'row_count', 'anomaly_count', 'normal_count'}
if not required_manifest_columns.issubset(split_manifest.columns):
    raise ValueError('Persisted building manifest is missing assignment or support fields')
if split_manifest.get_column('building_id').n_unique() != split_manifest.height:
    raise ValueError('Persisted manifest must contain one row per building')
manifest_hash = compute_file_sha256(split_path)
expected_hash = split_metadata.get('artifacts', {}).get('building_folds_sha256')
if manifest_hash != expected_hash:
    raise ValueError('Persisted building manifest does not match its recorded hash')
training_buildings = set(train_joined.select('building_id').unique().collect(engine='streaming')
                         .get_column('building_id').to_list())
test_buildings = set(test_joined.select('building_id').unique().collect(engine='streaming')
                     .get_column('building_id').to_list())
manifest_buildings = set(split_manifest.get_column('building_id').to_list())
if manifest_buildings != training_buildings:
    raise ValueError('Persisted manifest does not cover current training buildings exactly')
if manifest_buildings.intersection(test_buildings):
    raise ValueError('A test building appears in the training split manifest')
partition_support = (split_manifest.group_by(['split', 'fold'], maintain_order=True)
    .agg(pl.len().alias('buildings'), pl.col('row_count').sum().alias('rows'),
         pl.col('anomaly_count').sum().alias('anomaly_rows'),
         pl.col('normal_count').sum().alias('normal_rows'))
    .sort(['split', 'fold'], nulls_last=True))
if partition_support.filter((pl.col('anomaly_rows') == 0) | (pl.col('normal_rows') == 0)).height:
    raise ValueError('Every persisted partition must retain both target classes')
split_review = pl.DataFrame([{'manifest_buildings': split_manifest.height,
    'train_buildings_covered': len(training_buildings),
    'test_buildings_disjoint': not manifest_buildings.intersection(test_buildings),
    'seed_from_metadata': split_metadata.get('seed'),
    'manifest_sha256_prefix': manifest_hash[:12]}])
display(split_review)
display(partition_support)

manifest_buildings,train_buildings_covered,test_buildings_disjoint,seed_from_metadata,manifest_sha256_prefix
i64,i64,bool,i64,str
200,200,true,42,"""8cf86c278f82"""


split,fold,buildings,rows,anomaly_rows,normal_rows
str,i64,u32,i64,i64,i64
"""dev""",0,40,348967,7313,341654
"""dev""",1,40,349458,7682,341776
"""dev""",2,40,350767,7516,343251
"""dev""",3,40,350439,7545,342894
"""vault""",null,40,349863,7240,342623


**Observation:** The manifest summary and partition table are calculated from the current persisted assignment and training join.

**Interpretation:** Building-level assignment matches the unseen-building evaluation unit; it does not imply that a historical assignment is recoverable.

**Action:** Use this manifest as the fixed source for later validation. Do not replace it with a notebook-local split.

## Decisions for Notebook 02

### Decision ledger

| Decision | Current rule | Evidence boundary |
| --- | --- | --- |
| Problem formulation | Observed-hour point-anomaly classification | No future horizon is predicted |
| Target and primary metric | `anomaly`; ROC-AUC | Continuous scores preserve ranking |
| Grouping and validation | `building_id` is grouping-only; load the persisted manifest | Test buildings stay outside training folds |
| Feature provenance | `gte_*` is pending/blocked until verified | No name-based availability assumptions |
| Test-domain diagnostics | Shift diagnosis only | No test-guided feature selection or production-transform fitting |
| Data quality | Preserve labeled anomalies and unusual meter readings | No silent deletion or capping |
| Operational threshold | Not optimized | False-positive and false-negative costs are unspecified |

### Deferred experiments for Notebook 02

| Topic | Question to compare | Decision boundary |
| --- | --- | --- |
| Cleaning | Are any values demonstrable data defects? | Preserve rare and anomalous records unless a defect is verified |
| Missingness | Native missing values, imputation, and missingness indicators | Fit learned handling inside training folds |
| Preprocessing | Which model-specific transforms are needed? | Fit learned transforms on training folds only |
| Outlier policy | Retain raw readings versus a justified indicator or transform | Do not silently cap or delete; compare under fixed folds |
| Feature families and engineering | Which verified supplied or causal features add value? | Keep `gte_*` blocked until provenance and time availability are verified |
| PCA candidacy | Is a fold-local projection justified for a verified numeric block? | Compare with passthrough; do not assume dimensionality reduction helps |
| Imbalance | Do weighting or sampling alter ranking and guardrails? | Preserve ROC-AUC as primary; learned handling stays fold-local |

These are hypotheses for controlled comparisons, not conclusions about current data. Each applicable item should receive a finite experiment and an evidence-based outcome.

### Current evidence summary

The summary below is assembled from the target, quality, split, and domain-report values observed above.

**Purpose:** Generate a concise reader-facing summary from the values in the current notebook run.

In [7]:
def interpret_domain_auc(domain_auc: float | None) -> str:
    if domain_auc is None:
        return 'The current domain ROC-AUC is not available in the persisted summary.'
    if not 0.0 <= domain_auc <= 1.0:
        raise ValueError('Domain ROC-AUC must be within [0, 1]')
    if domain_auc < 0.5:
        return ('Pooled OOF ROC-AUC is below 0.5 for positive class `test=1`, '
                'indicating an inverse ranking. Inspect class/score orientation and '
                'fold-level metrics; do not invert scores after seeing this result.')
    if domain_auc <= 0.55:
        return 'The current score is in the weak-separation heuristic band.'
    if domain_auc <= 0.60:
        return 'The current score is in the moderate-separation heuristic band.'
    return 'The current score indicates more material domain separation under the heuristic.'

auc_values = [row['value'] for row in domain_metrics.to_dicts()
              if row['metric'].casefold().endswith('pooled_oof.roc_auc')]
domain_auc = auc_values[0] if auc_values else None
shift_interpretation = interpret_domain_auc(domain_auc)
positive_prevalence = float(target_summary.filter(pl.col('anomaly') == 1)
                            .get_column('row_share').item())
true_constant_count = feature_quality.filter(pl.col('status') == 'true constant').height
quasi_constant_count = feature_quality.filter(pl.col('status') == 'quasi-constant').height
manifest_building_count = split_review.get_column('manifest_buildings').item()
manifest_hash_prefix = split_review.get_column('manifest_sha256_prefix').item()
report_text = (
    f'**Observation:** Current positive-row share is {positive_prevalence:.4%}; '
    f'{true_constant_count} true constants and {quasi_constant_count} quasi-constants '
    f'are flagged for review. The persisted manifest covers {manifest_building_count} '
    f'training buildings and has hash prefix `{manifest_hash_prefix}`.\n\n'
    f'**Interpretation:** {shift_interpretation}\n\n'
    '**Action:** Retain the persisted building split, keep unverified feature families blocked, '
    'and defer learned preprocessing, outlier policy, and feature decisions to fixed-fold comparisons.'
)
display(Markdown(report_text))

**Observation:** Current positive-row share is 2.1318%; 2 true constants and 0 quasi-constants are flagged for review. The persisted manifest covers 200 training buildings and has hash prefix `8cf86c278f82`.

**Interpretation:** Pooled OOF ROC-AUC is below 0.5 for positive class `test=1`, indicating an inverse ranking. Inspect class/score orientation and fold-level metrics; do not invert scores after seeing this result.

**Action:** Retain the persisted building split, keep unverified feature families blocked, and defer learned preprocessing, outlier policy, and feature decisions to fixed-fold comparisons.